In [1]:
# -------------------------------
# KONFIGURATION
# -------------------------------
INITIAL_MODE = True  # Nur beim ersten Durchlauf auf True setzen!


In [1]:
import subprocess
import openpyxl
import os

# -------------------------------
# KONFIGURATION
# -------------------------------
datendump_gz = r"V:\06_DBSM\97_Skripte\DNBtitelundexemplare.dat.gz"
filtered_dat = r"C:\Users\sickerti\Projekte\inhaltsverzeichnis\I_Daten.dat"
auswertung_excel = r"C:\Users\sickerti\Projekte\inhaltsverzeichnis\Zwischenauswertung1.xlsx"
idn_file = r"C:\Users\sickerti\Projekte\inhaltsverzeichnis\idn.txt"
idn_temp_file = r"C:\Users\sickerti\Projekte\inhaltsverzeichnis\idn_neu.txt"

pica_filter_condition = "002@.0 =^ 'A' && 047I.c == '04'"

seen_in_run = set()


# -------------------------------
# Funktion zum Extrahieren von Unterfeldern
# -------------------------------
def extract_subfield(record, field_tag, subfield):
    fields = record.split("\x1e")  # Felder trennen
    for field in fields:
        field = field.strip()
        if field.startswith(field_tag):
            subfields = field.split("\x1f")
            for sf in subfields[1:]:
                if sf.startswith(subfield):
                    return sf[len(subfield):].strip()
    return None


def extract_subfields_all(record, field_tag, subfield):
    values = []
    fields = record.split("\x1e")
    for field in fields:
        field = field.strip()
        if field.startswith(field_tag):
            subfields = field.split("\x1f")
            for sf in subfields[1:]:
                if sf.startswith(subfield):
                    values.append(sf[len(subfield):].strip())
    return values

# -------------------------------
# Streaming-Iterator für PICA-Records
# -------------------------------
def iter_records(file_path):
    buffer = ""
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if line.startswith("001@") and buffer:
                yield buffer
                buffer = line
            else:
                buffer += line
        if buffer:
            yield buffer

# -------------------------------
# Schritt 1: PICA-Filter ausführen
# -------------------------------
print("Starte PICA-Filter...")
pica_cmd = ["pica", "filter", "-s", pica_filter_condition, datendump_gz]
with open(filtered_dat, 'w', encoding='utf-8') as outfile:
    subprocess.run(pica_cmd, stdout=outfile, check=True)
print(f"Gefilterte Datensätze wurden in {filtered_dat} gespeichert.")

# -------------------------------
# Schritt 2: vorhandene IDNs laden
# -------------------------------
existing_idn = set()
if os.path.exists(idn_file):
    with open(idn_file, 'r', encoding='utf-8') as f:
        existing_idn = {line.strip() for line in f if line.strip()}

# -------------------------------
# Schritt 3: Excel laden oder neu erstellen
# -------------------------------
headers = ["IDN", "Titel", "Titelzusatz", "Autor", "Ort", "Verlag", "Jahr", "DDC", "Schlagwort"]

import os
import openpyxl

# alte Datei löschen (wenn vorhanden)
if os.path.exists(auswertung_excel):
    os.remove(auswertung_excel)

# neue leere Excel erstellen
wb_aus = openpyxl.Workbook()
ws_aus = wb_aus.active

# Header setzen
ws_aus.append(headers)

# -------------------------------
# Schritt 4: Datensätze verarbeiten
# -------------------------------
print("Verarbeite Datensätze...")
processed_count = 0

for rec in iter_records(filtered_dat):
    idn = extract_subfield(rec, "003@", "0")
    if not idn:
        continue

    if idn in existing_idn or idn in seen_in_run:
        continue

    title = extract_subfield(rec, "021A", "a")
    zusatz = extract_subfield(rec, "021A", "d")
    author = extract_subfield(rec, "021A", "h")
    ort = extract_subfield(rec, "033A", "p")
    verlag = extract_subfield(rec, "033A", "n")
    jahr = extract_subfield(rec, "011@", "a")
    ddc = extract_subfield(rec, "045F", "a")
    schlag_list = extract_subfields_all(rec, "044N", "a")
    schlag = "; ".join(schlag_list) if schlag_list else None

    ws_aus.append([
        idn, title, zusatz, author,
        ort, verlag, jahr, ddc, schlag
    ])

    seen_in_run.add(idn)
    existing_idn.add(idn)

    

    processed_count += 1

# -------------------------------
# Schritt 5: Ergebnisse speichern
# -------------------------------
wb_aus.save(auswertung_excel)

# Excel-IDNs laden
excel_idns = set()

for row in ws_aus.iter_rows(min_row=2, max_col=1):
    val = row[0].value
    if val:
        excel_idns.add(str(val).strip())

# bestehende IDNs erneut laden (zur Sicherheit)
with open(idn_file, 'r', encoding='utf-8') as f:
    existing_check = {line.strip() for line in f if line.strip()}

# ❗ Prüfen: Excel darf KEINE alten IDNs enthalten
duplicates = excel_idns.intersection(existing_check)

if duplicates:
    print(f"⚠️ Entferne {len(duplicates)} doppelte IDNs aus Excel...")

    rows_to_delete = []

    for row in ws_aus.iter_rows(min_row=2):
        cell = row[0].value
        if cell and str(cell).strip() in duplicates:
            rows_to_delete.append(row[0].row)

    # Wichtig: von unten nach oben löschen!
    for r in reversed(rows_to_delete):
        ws_aus.delete_rows(r)

    wb_aus.save(auswertung_excel)

    print(f"✅ {len(rows_to_delete)} Zeilen aus Excel entfernt.")

# Excel nach Bereinigung neu lesen
final_excel_ids = set()

for row in ws_aus.iter_rows(min_row=2, max_col=1):
    val = row[0].value
    if val:
        final_excel_ids.add(str(val).strip())

# nur neue IDs anhängen
new_ids = final_excel_ids - existing_check

with open(idn_file, 'a', encoding='utf-8') as f:
    for i in new_ids:
        f.write(i + "\n")

print(f"✅ {len(new_ids)} neue IDNs angehängt.")

# Sicherheitsprüfung
with open(idn_file, 'r', encoding='utf-8') as f:
    idn_ids = {line.strip() for line in f if line.strip()}

missing = final_excel_ids - idn_ids

if missing:
    print(f"⚠️ {len(missing)} IDNs fehlen – werden ergänzt...")

    with open(idn_file, 'a', encoding='utf-8') as f:
        for m in missing:
            f.write(m + "\n")

print(f"Neue Datensätze: {processed_count}")
print(f"IDNs im aktuellen Lauf: {len(seen_in_run)}")
print(f"Fertig. {processed_count} neue Datensätze in Excel eingetragen.")


Starte PICA-Filter...
Gefilterte Datensätze wurden in C:\Users\sickerti\Projekte\inhaltsverzeichnis\I_Daten.dat gespeichert.
Verarbeite Datensätze...
✅ 6924 neue IDNs angehängt.
Neue Datensätze: 6924
IDNs im aktuellen Lauf: 6924
Fertig. 6924 neue Datensätze in Excel eingetragen.
